# 18. 최종 HS 코드 예측 모델 비교 및 안정성 검증

03번 TF-IDF + LinearSVC와 16번 Sentence-BERT 기반 과거 사례 검색을 동일한 144개 평가 사례와 동일한 4-fold StratifiedGroupKFold 분할에서 비교합니다.

비교 모델:
- TF-IDF + LinearSVC: 제품 속성 5개로 학습하는 분류 모델
- SBERT product_only: 제품 속성만 포함한 과거 사례 검색
- SBERT product_plus_ruling: 제품 속성 + 원본 판정요지 검색
- SBERT product_plus_ruling_stopword: 제품 속성 + 불용어 적용 판정요지 검색

검색 모델은 가장 유사한 과거 사례의 HS 코드를 Top-1 예측값으로 사용합니다. 모든 검색 문서는 해당 Fold의 학습 사례로만 구성합니다.

평가 지표는 Accuracy, Macro-F1, Weighted-F1, Top-3 Accuracy입니다. 제품 단위 paired bootstrap 신뢰구간과 모델 간 정답/오답 변화도 함께 확인합니다. 결과는 동일한 평가 사례에 대한 OOF(out-of-fold) 예측을 기준으로 계산합니다.

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy scipy

In [ ]:
import re,hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from kiwipiepy import Kiwi
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix
from scipy.stats import binomtest
from pathlib import Path

BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
DATA_FILE=quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
df=pd.read_csv(BASE+DATA_FILE,encoding='utf-8-sig').fillna('')
INPUT_COLUMNS=['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
TARGET='HS Code'
SUMMARY='Preprocessed Ruling Summary'
STOPWORDS={'것','경우','관세율표','규정','기재','등','따라서','물품','및','본건','본품','분류','설명','소호','수','이상','제품','함','해당','호','표','관세'}
for col in INPUT_COLUMNS+[TARGET,SUMMARY]: df[col]=df[col].astype(str).str.strip()
df=df[df[TARGET].ne('')].reset_index(drop=True)
df['model_text']='[PRODUCT] '+df['Product Name']+' [INGREDIENTS_KR] '+df['Main Ingredients (KR)']+' [INGREDIENTS_EN] '+df['Main Ingredients (EN)']+' [FORM] '+df['Product Form and Packaging']+' [USE] '+df['Intended Use']
df['query_text']=df[INPUT_COLUMNS].agg(' '.join,axis=1).str.strip()
df['duplicate_group']=pd.util.hash_pandas_object(df[INPUT_COLUMNS],index=False).astype(str)
class_counts=df[TARGET].value_counts()
eligible_labels=class_counts[class_counts>=4].index
eval_df=df[df[TARGET].isin(eligible_labels)].copy()
group_counts=eval_df.groupby(TARGET)['duplicate_group'].nunique()
eligible_labels=group_counts[group_counts>=4].index
eval_df=eval_df[eval_df[TARGET].isin(eligible_labels)].reset_index(drop=True)
X=eval_df['model_text'].reset_index(drop=True)
y=eval_df[TARGET].reset_index(drop=True)
groups=eval_df['duplicate_group'].reset_index(drop=True)
labels=np.sort(y.unique())
cv=StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=42)
splits=list(cv.split(X,y,groups))
print('평가 사례:',len(eval_df),'| HS 코드:',len(labels))
print('Fold 크기:',[len(te) for _,te in splits])

## 1. 불용어 전처리 및 임베딩 모델 준비

제품 속성 입력은 변경하지 않습니다. 불용어는 과거 판정요지에만 적용합니다. E5 임베딩은 16번과 동일하게 `intfloat/multilingual-e5-base`를 사용합니다.

In [ ]:
kiwi=Kiwi()
def remove_stopword_morphs(text):
    text=str(text)
    chars=list(text)
    for token in kiwi.tokenize(text):
        if token.form in STOPWORDS:
            for i in range(token.start,token.start+token.len): chars[i]=''
    return ''.join(chars)
eval_df['ruling_sw']=eval_df[SUMMARY].map(remove_stopword_morphs)
def make_vectorizer():
    return FeatureUnion([
        ('word',TfidfVectorizer(analyzer='word',ngram_range=(1,2),min_df=1,max_features=30000,sublinear_tf=True)),
        ('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(2,5),min_df=1,max_features=30000,sublinear_tf=True))
    ])
MODEL_NAME='intfloat/multilingual-e5-base'
sbert=SentenceTransformer(MODEL_NAME)
def encode_docs(texts):
    return sbert.encode(['passage: '+str(x) for x in texts],batch_size=32,show_progress_bar=False,normalize_embeddings=True,convert_to_numpy=True)
def encode_query(text):
    return sbert.encode(['query: '+str(text)],normalize_embeddings=True,convert_to_numpy=True)[0]
print('임베딩 모델:',MODEL_NAME,'| 장치:',sbert.device)

## 2. 동일 Fold에서 네 가지 모델 평가

TF-IDF + LinearSVC는 제품 속성 5개를 결합한 텍스트를 입력으로 사용합니다. SBERT 검색은 각 Fold의 학습 사례를 검색 문서로 사용하며, 평가 사례의 질의는 제품 속성 5개뿐입니다. Top-1 예측은 가장 유사한 학습 사례의 HS 코드입니다.

In [ ]:
prediction_rows=[]
fold_rows=[]
for fold,(tr,te) in enumerate(splits,1):
    train,test=eval_df.iloc[tr].copy(),eval_df.iloc[te].copy()
    fold_rows.append({'fold':fold,'train_n':len(tr),'test_n':len(te),'group_overlap':len(set(groups.iloc[tr])&set(groups.iloc[te]))})
    vectorizer=make_vectorizer()
    x_train=vectorizer.fit_transform(X.iloc[tr])
    x_test=vectorizer.transform(X.iloc[te])
    svm=LinearSVC(C=1.0,class_weight='balanced').fit(x_train,y.iloc[tr])
    svm_pred=svm.predict(x_test)
    svm_scores=svm.decision_function(x_test)
    svm_classes=svm.classes_
    for local_i,global_i in enumerate(te):
        ranked=svm_classes[np.argsort(svm_scores[local_i])[::-1]]
        prediction_rows.append({'row_index':int(global_i),'fold':fold,'true_hs':y.iloc[global_i],'model':'TFIDF_LinearSVC','predicted_hs':svm_pred[local_i],'top3_candidates':list(ranked[:3])})
    doc_texts={
        'SBERT_product_only':train['query_text'].tolist(),
        'SBERT_product_plus_ruling':(train['query_text']+' '+train[SUMMARY]).str.strip().tolist(),
        'SBERT_product_plus_ruling_stopword':(train['query_text']+' '+train['ruling_sw']).str.strip().tolist()
    }
    doc_embeddings={name:encode_docs(texts) for name,texts in doc_texts.items()}
    for global_i in te:
        row=eval_df.iloc[global_i]
        q=encode_query(row['query_text'])
        for name,emb in doc_embeddings.items():
            scores=emb@q
            order=np.argsort(scores)[::-1]
            top3_codes=[]
            for j in order:
                code=train.iloc[j][TARGET]
                if code not in top3_codes: top3_codes.append(code)
                if len(top3_codes)==3: break
            prediction_rows.append({'row_index':int(global_i),'fold':fold,'true_hs':row[TARGET],'model':name,'predicted_hs':train.iloc[order[0]][TARGET],'top3_candidates':top3_codes})
predictions=pd.DataFrame(prediction_rows)
fold_audit=pd.DataFrame(fold_rows)
display(fold_audit)
if fold_audit['group_overlap'].sum()!=0: raise ValueError('Train/test 그룹 중복 발생')
print('OOF 예측 수:',len(predictions),'| 모델별 예측 수:',predictions.groupby('model').size().to_dict())

## 3. 전체 OOF 성능 비교

모든 평가 사례는 자신이 학습에 포함되지 않은 Fold의 모델로 예측됩니다. Macro-F1은 평가 대상 13개 HS 코드를 고정해 계산합니다. Top-3 Accuracy는 분류 모델의 상위 3개 후보 또는 검색 모델에서 서로 다른 HS 코드 기준 상위 3개 후보에 정답 코드가 포함되는 비율입니다.

In [ ]:
def metric_summary(part):
    yt=part['true_hs'].astype(str)
    yp=part['predicted_hs'].astype(str)
    top3=np.mean([str(t) in [str(x) for x in c] for t,c in zip(yt,part['top3_candidates'])])
    return {'n':len(part),'accuracy':accuracy_score(yt,yp),'macro_f1':f1_score(yt,yp,labels=labels,average='macro',zero_division=0),'weighted_f1':f1_score(yt,yp,average='weighted',zero_division=0),'top3_accuracy':top3}
summary=pd.DataFrame([{'model':name,**metric_summary(part)} for name,part in predictions.groupby('model')]).sort_values('macro_f1',ascending=False)
display(summary.round(4))
fold_metrics=[]
for (model_name,fold),part in predictions.groupby(['model','fold']):
    fold_metrics.append({'model':model_name,'fold':fold,**metric_summary(part)})
fold_metrics=pd.DataFrame(fold_metrics)
display(fold_metrics.pivot(index='fold',columns='model',values='accuracy').round(4))
display(fold_metrics.pivot(index='fold',columns='model',values='macro_f1').round(4))

## 4. Paired bootstrap 신뢰구간

동일한 평가 사례에서 예측된 결과를 기준으로 2,000회 paired bootstrap을 수행합니다. 각 반복에서 평가 사례를 복원추출하고 Accuracy, Macro-F1, Weighted-F1을 계산해 95% percentile 신뢰구간을 구합니다. 이는 평가 표본에 대한 불확실성 추정이며, 외부 데이터에 대한 일반화 성능을 보장하지 않습니다.

In [ ]:
rng=np.random.default_rng(42)
N_BOOT=2000
bootstrap_rows=[]
wide_true=predictions.pivot(index='row_index',columns='model',values='predicted_hs')
true_by_row=predictions.drop_duplicates('row_index').set_index('row_index')['true_hs'].astype(str)
top3_by_model={name:predictions[predictions['model'].eq(name)].set_index('row_index')['top3_candidates'] for name in predictions['model'].unique()}
row_ids=wide_true.index.to_numpy()
for model_name in wide_true.columns:
    pred=wide_true[model_name].astype(str).to_numpy()
    true=true_by_row.loc[row_ids].to_numpy()
    vals={'accuracy':[],'macro_f1':[],'weighted_f1':[]}
    for _ in range(N_BOOT):
        sample=rng.integers(0,len(row_ids),len(row_ids))
        vals['accuracy'].append(accuracy_score(true[sample],pred[sample]))
        vals['macro_f1'].append(f1_score(true[sample],pred[sample],labels=labels,average='macro',zero_division=0))
        vals['weighted_f1'].append(f1_score(true[sample],pred[sample],labels=labels,average='weighted',zero_division=0))
    for metric,arr in vals.items():
        bootstrap_rows.append({'model':model_name,'metric':metric,'estimate':float(np.mean(arr)),'ci_2.5':float(np.quantile(arr,.025)),'ci_97.5':float(np.quantile(arr,.975))})
bootstrap_ci=pd.DataFrame(bootstrap_rows)
display(bootstrap_ci.round(4))

## 5. 모델 간 예측 차이

각 모델 쌍에 대해 한 모델만 정답을 맞힌 사례와 다른 모델만 정답을 맞힌 사례를 비교합니다. McNemar exact binomial test는 두 모델의 오류 패턴 차이를 확인하는 참고 검정입니다. 여러 쌍을 비교하므로 p-value를 확정적 결론으로 과해석하지 않습니다.

In [ ]:
from itertools import combinations
comparison_rows=[]
truth=true_by_row.loc[row_ids].astype(str)
for a,b in combinations(wide_true.columns,2):
    pa=wide_true[a].astype(str)
    pb=wide_true[b].astype(str)
    a_correct=pa.eq(truth)
    b_correct=pb.eq(truth)
    a_only=int((a_correct&~b_correct).sum())
    b_only=int((~a_correct&b_correct).sum())
    p=binomtest(min(a_only,b_only),a_only+b_only,.5).pvalue if a_only+b_only else 1.0
    comparison_rows.append({'model_a':a,'model_b':b,'both_correct':int((a_correct&b_correct).sum()),'a_only_correct':a_only,'b_only_correct':b_only,'both_wrong':int((~a_correct&~b_correct).sum()),'mcnemar_exact_p':p})
paired_tests=pd.DataFrame(comparison_rows)
display(paired_tests.round(4))
class_rows=[]
for model_name,part in predictions.groupby('model'):
    report=classification_report(part['true_hs'].astype(str),part['predicted_hs'].astype(str),labels=labels,output_dict=True,zero_division=0)
    for code in labels:
        class_rows.append({'model':model_name,'hs_code':code,'precision':report[str(code)]['precision'],'recall':report[str(code)]['recall'],'f1':report[str(code)]['f1-score'],'support':report[str(code)]['support']})
class_metrics=pd.DataFrame(class_rows)
display(class_metrics.pivot(index='hs_code',columns='model',values='f1').round(3))

## 6. 결과 해석 시 주의사항

- TF-IDF + LinearSVC는 텍스트 분류 모델이고, SBERT는 학습 Fold의 과거 사례를 검색해 HS 코드를 가져오는 검색 기반 예측 방식입니다.
- 동일한 144개 평가 사례와 동일한 Fold를 사용하므로 직접 비교가 가능합니다.
- 검색 모델의 Top-1은 가장 유사한 과거 사례의 코드이며, Top-3는 상위 검색 사례에서 중복 코드를 제거한 후보입니다.
- Macro-F1은 소수 클래스 성능에 민감하므로 Accuracy 및 Weighted-F1과 함께 해석합니다.
- 이번 비교는 기존 사례 데이터 내부의 교차검증입니다. 실제 신규 제품이나 다른 기간·출처의 판정 사례에 대한 외부 검증은 별도 과제입니다.

In [ ]:
OUT=Path('/content/hs_final_model_comparison')
OUT.mkdir(parents=True,exist_ok=True)
predictions.to_csv(OUT/'18_oof_predictions.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'18_model_comparison.csv',index=False,encoding='utf-8-sig')
fold_metrics.to_csv(OUT/'18_fold_metrics.csv',index=False,encoding='utf-8-sig')
bootstrap_ci.to_csv(OUT/'18_bootstrap_ci.csv',index=False,encoding='utf-8-sig')
paired_tests.to_csv(OUT/'18_paired_model_tests.csv',index=False,encoding='utf-8-sig')
class_metrics.to_csv(OUT/'18_class_metrics.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv(OUT/'18_fold_audit.csv',index=False,encoding='utf-8-sig')
print('18번 결과 저장 완료:',OUT)
import shutil
from google.colab import files
shutil.make_archive('/content/18_final_model_comparison_results','zip',OUT)
files.download('/content/18_final_model_comparison_results.zip')